# 14 Perplexity as a gate: what a small language model's surprise can and cannot flag

A gate is a score, a threshold set before the run, and an action. The scores so far came from classifiers. A small language model gives a different score for free: **perplexity**, how surprised the model is by a text, computed as the exponential of the average negative log-probability it assigns to each token. A message the model finds ordinary has low perplexity; gibberish and odd formatting have high perplexity. This notebook asks whether that single forward pass can gate three kinds of unusual Meridian input: fluent prompt injections, word-swap attacks from notebook 09, and out-of-scope messages.

**The gate rule.** Choose the threshold from benign messages only, before looking at any attack: the perplexity that 5% of benign training messages exceed. Then count how many unusual messages exceed it. That fixes the false-alarm budget in advance, as the conformal gate in notebook 03 does.

**What the numbers cover.** A 0.5B base language model (Qwen2.5-0.5B) on CPU; the 200 Meridian test messages; the 96 injected messages from notebook 08 (written by one language model, fluent English, this attack style only); the word-swap attack texts from notebook 09 that fooled the encoder; and the out-of-scope test messages. Intervals are Wilson for counts and a bootstrap over messages for AUROC. Messages are variants of 40 test scenarios, so intervals on the benign group are too narrow.

In [1]:
import json, time
import numpy as np, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score
import meridian as m

torch.set_num_threads(3)
NAME = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(NAME, local_files_only=True)
lm = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32, local_files_only=True).eval()

def nll(text):
    """Mean negative log-probability per token: the log of perplexity."""
    ids = tok(text, return_tensors="pt").input_ids
    x = torch.cat([torch.tensor([[tok.eos_token_id]]), ids], 1)   # the end-of-text token gives the first token a context
    with torch.inference_mode():
        lp = torch.log_softmax(lm(input_ids=x).logits[0, :-1].float(), -1).gather(1, x[0, 1:, None])[:, 0]
    return float(-lp.mean())

tr, te = m.load("train"), m.load("test")
inj = [json.loads(l) for l in open("data/llm/injection_messages.jsonl")]
att = [a for a in map(json.loads, open("data/cache/09_attack_log.jsonl")) if a["success"]]
print(f"{len(te)} benign test | {len(inj)} injected | {len(att)} word-swap attacks that fooled the encoder")

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


200 benign test | 96 injected | 171 word-swap attacks that fooled the encoder


In [2]:
t0 = time.perf_counter()
sub = tr[::4]                                                   # every fourth training message sets the threshold
nll_train = np.array([nll(r["text"]) for r in sub])
thr = float(np.quantile(nll_train, 0.95))
nll_test = np.array([nll(r["text"]) for r in te])
nll_inj = np.array([nll(r["text"]) for r in inj])
nll_orig = np.array([nll(a["text"]) for a in att])
nll_adv = np.array([nll(a["adv"]) for a in att])
per_msg = (time.perf_counter() - t0) / (len(sub) + len(te) + len(inj) + 2 * len(att))
print(f"threshold (95th percentile of {len(sub)} benign training messages): mean token surprise {thr:.2f}, perplexity {np.exp(thr):.1f}")
print(f"scoring took {per_msg * 1000:.0f} ms per message on this machine")

threshold (95th percentile of 200 benign training messages): mean token surprise 6.22, perplexity 502.8
scoring took 630 ms per message on this machine


**Goal.** Score every message once and fix the threshold from benign training messages alone.

## Detection at the threshold chosen in advance

For each group of unusual messages the table gives the share above the threshold with a Wilson interval, next to the false-alarm rate on benign test messages. AUROC says how well the score ranks the unusual group above benign messages regardless of threshold: 0.5 is a coin flip and 1.0 is perfect.

In [3]:
def auroc(neg, pos, B=2000, seed=0):
    y = np.r_[np.zeros(len(neg)), np.ones(len(pos))]; s = np.r_[neg, pos]
    rng = np.random.default_rng(seed); a = roc_auc_score(y, s); bs = []
    for _ in range(B):
        i = rng.integers(0, len(neg), len(neg)); j = rng.integers(0, len(pos), len(pos))
        bs.append(roc_auc_score(np.r_[np.zeros(len(i)), np.ones(len(j))], np.r_[neg[i], pos[j]]))
    return a, float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

oos = np.array([r["label"] == "out_of_scope" for r in te])
groups = {"benign test messages (false alarms)": nll_test,
          "fluent prompt injections": nll_inj,
          "word-swap attacks that fooled the encoder": nll_adv,
          "out-of-scope messages": nll_test[oos]}
res = {}
print(f"{'group':44s}{'n':>5s}{'above threshold':>18s}{'Wilson 95%':>18s}{'AUROC vs benign [95% bootstrap]':>34s}")
for name, v in groups.items():
    k = int((v > thr).sum()); lo, hi = m.wilson(k, len(v))
    if name.startswith("benign"):
        res[name] = {"n": len(v), "k": k}; print(f"{name:44s}{len(v):>5d}{k:>10d} ({k / len(v):>5.1%}){f'[{lo:.3f}, {hi:.3f}]':>18s}"); continue
    neg = nll_test[~oos] if name.startswith("out-of") else nll_test
    a, alo, ahi = auroc(neg, v)
    res[name] = {"n": len(v), "k": k, "auroc": a, "auroc_ci": [alo, ahi]}
    print(f"{name:44s}{len(v):>5d}{k:>10d} ({k / len(v):>5.1%}){f'[{lo:.3f}, {hi:.3f}]':>18s}{f'{a:.3f} [{alo:.3f}, {ahi:.3f}]':>34s}")
a, alo, ahi = auroc(nll_orig, nll_adv)
print(f"\npaired: the same {len(att)} messages before and after the word swaps, AUROC {a:.3f} [{alo:.3f}, {ahi:.3f}]; surprise rose on {int((nll_adv > nll_orig).sum())} of {len(att)}")
res["paired_swap"] = {"auroc": a, "ci": [alo, ahi], "n_up": int((nll_adv > nll_orig).sum()), "n": len(att)}

group                                           n   above threshold        Wilson 95%   AUROC vs benign [95% bootstrap]
benign test messages (false alarms)           200        16 ( 8.0%)    [0.050, 0.126]


fluent prompt injections                       96         0 ( 0.0%)    [0.000, 0.038]              0.422 [0.360, 0.484]


word-swap attacks that fooled the encoder     171        44 (25.7%)    [0.198, 0.328]              0.767 [0.716, 0.813]


out-of-scope messages                          25         4 (16.0%)    [0.064, 0.347]              0.501 [0.369, 0.629]



paired: the same 171 messages before and after the word swaps, AUROC 0.734 [0.682, 0.785]; surprise rose on 166 of 171


**Goal.** Count how many unusual messages exceed a threshold chosen from benign messages alone, and how well the score ranks them above benign ones.

**What we got.** The threshold, set for a 5% false-alarm rate on benign training messages, fired on 16 of 200 benign test messages (8.0%, Wilson interval 5.0% to 12.6%), so the budget was exceeded on the test split. It flagged 0 of 96 fluent injections (Wilson upper bound 3.8%) and ranks them below benign messages (AUROC 0.42, interval 0.36 to 0.48): the injected messages were less surprising than ordinary ones, because they are well-formed English built from common phrases. It flagged 44 of 171 word-swap attacks (25.7%, Wilson 19.8% to 32.8%; AUROC 0.77). On the same 171 messages, surprise rose after the swap on 166, so the signal is there within a message; but ordinary messages differ in surprise by more than a swap adds, and a gate that sees only the new message cannot compare it with an original. Out-of-scope messages are no better than chance (AUROC 0.50, interval 0.37 to 0.63, 25 messages).

## Perplexity plus length

A short message is surprising for a language model almost by default: the first tokens have little context. Alon and Kamfonas (2023) found that plain perplexity filtering of adversarial suffixes gives many false positives and that a gradient-boosted classifier on perplexity and token length fixed them (https://arxiv.org/abs/2308.14132). The cell below checks whether the message length explains the surprise on Meridian, and whether adding it changes what the gate flags.

In [4]:
from sklearn.linear_model import LogisticRegression
ntok = lambda t: len(tok(t).input_ids)
len_test = np.array([ntok(r["text"]) for r in te]); len_inj = np.array([ntok(r["text"]) for r in inj]); len_adv = np.array([ntok(a["adv"]) for a in att])
print(f"tokens per message: benign median {np.median(len_test):.0f}, injected median {np.median(len_inj):.0f}, word-swapped median {np.median(len_adv):.0f}")
print(f"correlation of surprise with length on benign test messages: {np.corrcoef(nll_test, len_test)[0, 1]:+.2f}")
def auc_len(neg_s, neg_l, pos_s, pos_l):
    X = np.c_[np.r_[neg_s, pos_s], np.r_[neg_l, pos_l]]; y = np.r_[np.zeros(len(neg_s)), np.ones(len(pos_s))]
    return roc_auc_score(y, LogisticRegression(max_iter=1000).fit(X, y).decision_function(X))
print(f"fluent injections, length alone: AUROC {roc_auc_score(np.r_[np.zeros(len(te)), np.ones(len(inj))], np.r_[len_test, len_inj]):.3f}")
print(f"fluent injections, surprise alone {res['fluent prompt injections']['auroc']:.3f}; surprise and length (fit and scored on the same messages, so optimistic): {auc_len(nll_test, len_test, nll_inj, len_inj):.3f}")

tokens per message: benign median 39, injected median 37, word-swapped median 40
correlation of surprise with length on benign test messages: -0.70
fluent injections, length alone: AUROC 0.525
fluent injections, surprise alone 0.422; surprise and length (fit and scored on the same messages, so optimistic): 0.589


**What we got.** Surprise falls as messages get longer (correlation -0.70 on benign test messages): a short message gives the model little context, so shortness alone raises perplexity, the source of false positives that Alon and Kamfonas describe. The three groups have similar lengths (medians 39, 37 and 40 tokens). Length alone ranks injections at chance (AUROC 0.525), and surprise with length reaches 0.589, fitted and scored on the same messages and still near chance.

In [5]:
out = {"model": NAME, "threshold_nll": thr, "threshold_perplexity": float(np.exp(thr)), "n_train_threshold": len(sub),
       "ms_per_message": per_msg * 1000, "groups": res}
Path("data/results/14_perplexity_gate.json").write_text(json.dumps(out, indent=2, default=float))

1027

**What this shows for the pattern.** Perplexity is a one-pass gate for text that is statistically odd. It partly catches word-swapped messages (about a quarter at an 8% false-alarm rate) and, in the cited literature, adversarial gibberish suffixes. It does not catch text that is fluent and hostile, and it says nothing about whether a request is in scope. It is also not cheap next to the encoder: about 0.6 s per message here for a 0.5B model on a loaded laptop, against about 31 ms for an encoder pass in notebook 11. Use it as one input beside the encoder's confidence and a trained injection classifier (notebook 08's flagged 56 of 96 at 11 false alarms in 200), not as the injection defense. Set its threshold from benign messages, as here, and check the false-alarm rate on held-out benign messages: the 5% budget came out at 8%.

The same family of one-pass scores includes the cross-encoder relevance score that decides whether any retrieved passage is good enough to answer from, and the token log-probability of an answer; the LLM tier chapter and the abstention chapter take those up. For training text, a perplexity filter against a reference corpus keeps documents close to that corpus (CCNet, https://arxiv.org/abs/1911.00359), which selects a register, not quality.